# Empirical Paper Pipeline — Colab Runner

Run the full analysis pipeline on Colab to avoid crashing local hardware.

## What to upload

Use the **Files** panel (folder icon, left sidebar) to upload **two things**:

1. `events.csv` — from `analysis/data/events.csv` (~20 MB)
2. `analysis_code.zip` — built by the cell below from your local repo, **or** upload manually

Everything else is installed/generated here.

---
## 0. Install dependencies

In [ ]:
!pip install -q pandas numpy scipy statsmodels scikit-learn matplotlib seaborn \
    pyyaml jinja2 tqdm tabulate hmmlearn pyEDM joblib

---
## 1. Upload files

Run this cell, then use the dialog to upload:
- `events.csv`
- `analysis_code.zip` (created by the helper script below, or zipped manually)

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload events.csv and analysis_code.zip

---
## 2. Unpack code and set up directory structure

In [ ]:
import os
import zipfile
import shutil

# Unzip the analysis code
with zipfile.ZipFile("analysis_code.zip", "r") as z:
    z.extractall("/content/analysis")

# Set up data directory and move events.csv into it
os.makedirs("/content/analysis/data", exist_ok=True)
shutil.move("events.csv", "/content/analysis/data/events.csv")

# Verify
print("Code files:")
for root, dirs, fnames in os.walk("/content/analysis/pipeline"):
    level = root.replace("/content/analysis/pipeline", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in sorted(fnames):
        if f.endswith(".py"):
            print(f"{indent}  {f}")

print(f"\nData: {os.path.getsize('/content/analysis/data/events.csv') / 1e6:.1f} MB")

---
## 3. Write Colab-tuned config

Same as your local `config.yaml` but with:
- `data_source: local` (no Firebase needed)
- Higher parallelism (Colab has the RAM for it)

In [ ]:
config_text = """\
# Colab-tuned config
data_source: local
local_data_dir: ./data
output_dir: ./outputs
plot_format: png
dpi: 150

min_clicks_per_session: 50
min_session_duration_s: 300
exclude_test_sessions: true

phase_boundaries:
  - {id: 1, start_ms: 0, end_ms: 90000, label: "Symmetric baseline"}
  - {id: 2, start_ms: 90000, end_ms: 180000, label: "A advantage"}
  - {id: 3, start_ms: 180000, end_ms: 270000, label: "B advantage"}
  - {id: 4, start_ms: 270000, end_ms: 360000, label: "Scarcity + pulses"}

bonus_pulses:
  phase4_start_ms: 270000
  interval_ms: 12000
  duration_ms: 3000
  sequence: [A, B, A, B, A, B, A, B]

rolling_window_time_s: 15
rolling_window_clicks: 20
smoothing_window_clicks: 10

transition_pre_window_s: 15
transition_post_window_s: 30
adaptation_threshold: 0.6

pulse_pre_window_s: 10
pulse_post_window_s: 15

n_bootstrap: 1000

run_models: true
model_families:
  - baseline
  - matching
  - rl
  - hmm

rl_models:
  n_starts: 10

n_simulations_per_model: 50

run_clustering: true
n_clusters_range: [2, 5]

generate_report: true
report_format: html
"""

with open("/content/analysis/config.yaml", "w") as f:
    f.write(config_text)

print("Config written.")

---
## 4. Patch parallelism for Colab resources

Colab free tier has ~12 GB RAM and 2 CPU cores.  
Colab Pro has ~25 GB and up to 4 cores.  
We set `n_jobs=2, batch_size=8` which is safe for the free tier.

In [ ]:
import psutil

ram_gb = psutil.virtual_memory().total / (1024 ** 3)
cpu_count = os.cpu_count()
print(f"Colab runtime: {ram_gb:.1f} GB RAM, {cpu_count} CPUs")

# Set parallelism based on available RAM
if ram_gb > 20:
    n_jobs, batch_size = 4, 16
    tier = "Pro (high RAM)"
elif ram_gb > 10:
    n_jobs, batch_size = 2, 8
    tier = "Free tier"
else:
    n_jobs, batch_size = 1, 4
    tier = "Low RAM"

print(f"Detected: {tier} -> n_jobs={n_jobs}, batch_size={batch_size}")

# Patch dynamical_analysis.py
dyn_path = "/content/analysis/pipeline/dynamical_analysis.py"
with open(dyn_path, "r") as f:
    code = f.read()

code = code.replace(
    '_MAX_JOBS = 1  # sequential — avoids memory pressure on laptops',
    f'_MAX_JOBS = {n_jobs}  # auto-tuned for Colab'
)
code = code.replace(
    '_BATCH_SIZE = 2  # small batches to limit peak memory',
    f'_BATCH_SIZE = {batch_size}  # auto-tuned for Colab'
)

with open(dyn_path, "w") as f:
    f.write(code)

print(f"Patched dynamical_analysis.py: _MAX_JOBS={n_jobs}, _BATCH_SIZE={batch_size}")

---
## 5. Run the full pipeline

In [ ]:
os.chdir("/content/analysis")

# Set thread limits to avoid BLAS oversubscription
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"] = "2"
os.environ["OPENBLAS_NUM_THREADS"] = "2"
os.environ["VECLIB_MAXIMUM_THREADS"] = "2"

!python run.py 2>&1 | tee pipeline_log.txt

---
## 6. Download results

Zips up all outputs and triggers a browser download.

In [ ]:
# Show what was generated
!find /content/analysis/outputs -type f | head -30
!echo "..."
!find /content/analysis/outputs -type f | wc -l
print("files total")

In [ ]:
# Zip and download
shutil.make_archive("/content/pipeline_outputs", "zip", "/content/analysis/outputs")
print(f"Archive size: {os.path.getsize('/content/pipeline_outputs.zip') / 1e6:.1f} MB")

files.download("/content/pipeline_outputs.zip")

---
## Optional: Run subsets

If the full pipeline is too slow or you want to iterate on parts:

In [ ]:
# Models only (skip NDS diagnostics — the slow part)
# !python run.py --models-only

# NDS only (skip model fitting)
# !python run.py --skip-models

# Quick test with 5 sessions
# !python run.py --n-sessions 5